# Stand-up Comedy — Text Mining

Análisis end-to-end sobre los transcripts de stand-up cruzados con
metadata de IMDb. Todo el notebook está parametrizado por **filtros**
(comediante, rango de años, rating mínimo, top-N por votos) — cambia
los filtros en la sección "Filtros" y vuelve a correr para regenerar
las visualizaciones sobre el subset elegido.


## 1. Setup

In [ ]:
import re
import warnings
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
import nltk
import numpy as np
import pandas as pd
import seaborn as sns
from nltk.corpus import stopwords
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
from wordcloud import WordCloud

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")
pd.set_option("display.max_colwidth", 80)

for pkg in ["stopwords", "punkt", "punkt_tab"]:
    try:
        nltk.data.find(f"corpora/{pkg}" if pkg == "stopwords" else f"tokenizers/{pkg}")
    except LookupError:
        nltk.download(pkg, quiet=True)

STOPWORDS = set(stopwords.words("english")) | {
    "im", "youre", "thats", "dont", "didnt", "doesnt", "ive", "youve",
    "hes", "shes", "theyre", "weve", "wouldnt", "couldnt", "shouldnt",
    "wont", "cant", "isnt", "arent", "wasnt", "werent", "gonna", "wanna",
    "gotta", "yeah", "okay", "ok", "uh", "um", "like", "know", "right",
    "well", "one", "get", "got", "going", "go", "say", "said", "really",
    "people", "think", "thing", "things", "make", "made", "way",
}


## 2. Cargar y limpiar datos

In [ ]:
DATA = Path("../data/data_frame/df_unified.parquet")
df_all = pd.read_parquet(DATA).copy()

# La columna `comedian` a veces trae "Stand-up transcripts" — fallback al título.
def comedian_from_title(t):
    if not isinstance(t, str):
        return None
    base = re.split(r"[:\-–]", t, maxsplit=1)[0]
    return base.strip().title()

mask_bad = df_all["comedian"].isin(["Stand-up transcripts", None, ""]) | df_all["comedian"].isna()
df_all.loc[mask_bad, "comedian"] = df_all.loc[mask_bad, "title"].map(comedian_from_title)

# Runtime: viene como lista [n] o None → escalar minutos.
df_all["runtime_min"] = df_all["runtimes"].map(
    lambda x: int(x[0]) if isinstance(x, (list, np.ndarray)) and len(x) > 0 else np.nan
)
df_all["year"] = pd.to_numeric(df_all["year"], errors="coerce")
df_all["rating"] = pd.to_numeric(df_all["rating"], errors="coerce")
df_all["votes"] = pd.to_numeric(df_all["votes"], errors="coerce")
df_all["word_count"] = df_all["transcript"].str.split().str.len()

print(f"Shows totales: {len(df_all)}")
print(f"Comediantes únicos: {df_all['comedian'].nunique()}")
print(f"Rango años: {int(df_all['year'].min())}–{int(df_all['year'].max())}")
df_all[["title", "comedian", "year", "rating", "votes", "runtime_min", "word_count"]].head()


## 3. Filtros

Cambia los valores y vuelve a correr **desde aquí hacia abajo** para
regenerar el análisis sobre el subset.


In [ ]:
# === EDITA AQUÍ ===
COMEDIANS = None           # ej: ["Dave Chappelle", "Bo Burnham"] o None para todos
YEAR_RANGE = (2000, 2025)  # (min, max) inclusive
MIN_RATING = 0.0           # ej: 7.5 para "bien valorados"
MIN_VOTES = 0              # filtro de confiabilidad del rating
TOP_N_BY_VOTES = None      # ej: 20 para "los 20 más populares"; None = sin tope
# ==================

def apply_filters(df):
    out = df.copy()
    if COMEDIANS:
        out = out[out["comedian"].isin(COMEDIANS)]
    out = out[(out["year"].between(*YEAR_RANGE)) | out["year"].isna()]
    out = out[out["rating"].fillna(0) >= MIN_RATING]
    out = out[out["votes"].fillna(0) >= MIN_VOTES]
    if TOP_N_BY_VOTES:
        out = out.nlargest(TOP_N_BY_VOTES, "votes")
    return out.reset_index(drop=True)

df = apply_filters(df_all)
print(f"Shows tras filtros: {len(df)} (de {len(df_all)})")
print(f"Comediantes: {df['comedian'].nunique()}")
df[["title", "comedian", "year", "rating", "votes"]].head(10)


## 4. EDA

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 8))
df["word_count"].hist(bins=30, ax=axes[0, 0]); axes[0, 0].set_title("Palabras por show")
df["rating"].dropna().hist(bins=20, ax=axes[0, 1]); axes[0, 1].set_title("Rating IMDb")
df["year"].dropna().astype(int).value_counts().sort_index().plot.bar(ax=axes[1, 0]); axes[1, 0].set_title("Shows por año")
df["runtime_min"].dropna().hist(bins=20, ax=axes[1, 1]); axes[1, 1].set_title("Runtime (min)")
plt.tight_layout(); plt.show()

df["words_per_min"] = df["word_count"] / df["runtime_min"]
top_talkers = df.dropna(subset=["words_per_min"]).nlargest(15, "words_per_min")[
    ["comedian", "title", "words_per_min"]
]
top_talkers


## 5. Tokenización y riqueza léxica

In [ ]:
TOKEN_RE = re.compile(r"[a-z]+")

def tokenize(text):
    return [t for t in TOKEN_RE.findall(text.lower()) if t not in STOPWORDS and len(t) > 2]

df["tokens"] = df["transcript"].fillna("").map(tokenize)
df["unique_words"] = df["tokens"].map(lambda t: len(set(t)))
df["ttr"] = df["unique_words"] / df["tokens"].map(len).replace(0, np.nan)

lex = df.groupby("comedian").agg(
    shows=("title", "count"),
    avg_words=("word_count", "mean"),
    avg_ttr=("ttr", "mean"),
    avg_rating=("rating", "mean"),
).sort_values("avg_ttr", ascending=False).head(15)
lex


## 6. N-gramas más frecuentes

In [ ]:
def top_ngrams(corpus, n=1, top_k=20):
    vec = CountVectorizer(ngram_range=(n, n), stop_words=list(STOPWORDS), min_df=2)
    X = vec.fit_transform(corpus)
    sums = X.sum(axis=0).A1
    return pd.Series(sums, index=vec.get_feature_names_out()).nlargest(top_k)

corpus = df["transcript"].fillna("").tolist()

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
for ax, n, title in zip(axes, [1, 2, 3], ["Unigramas", "Bigramas", "Trigramas"]):
    top_ngrams(corpus, n=n).sort_values().plot.barh(ax=ax)
    ax.set_title(f"Top {title}")
plt.tight_layout(); plt.show()


## 7. WordCloud

In [ ]:
text_blob = " ".join(" ".join(t) for t in df["tokens"])
wc = WordCloud(width=1200, height=600, background_color="white",
               max_words=200, collocations=False).generate(text_blob)
plt.figure(figsize=(14, 7))
plt.imshow(wc, interpolation="bilinear")
plt.axis("off")
plt.title("WordCloud — subset filtrado")
plt.show()


## 8. Sentimiento (VADER)

In [ ]:
sia = SentimentIntensityAnalyzer()

def sentiment_score(text):
    return sia.polarity_scores(text or "")["compound"]

df["sentiment"] = df["transcript"].map(sentiment_score)

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
df["sentiment"].hist(bins=30, ax=axes[0]); axes[0].set_title("Sentimiento por show (compound)")
axes[0].axvline(0, color="red", linestyle="--")

by_com = df.groupby("comedian")["sentiment"].mean().sort_values()
by_com.tail(20).plot.barh(ax=axes[1]); axes[1].set_title("Sentimiento promedio por comediante")
plt.tight_layout(); plt.show()


### 8.1 Evolución del sentimiento dentro de un show

In [ ]:
# Elige uno del subset
SHOW_TITLE = df.iloc[0]["title"]  # cámbialo a uno específico si quieres
SEGMENTS = 20

row = df[df["title"] == SHOW_TITLE].iloc[0]
sents = nltk.sent_tokenize(row["transcript"])
chunks = np.array_split(sents, SEGMENTS)
scores = [np.mean([sia.polarity_scores(s)["compound"] for s in c]) if len(c) else 0 for c in chunks]

plt.figure(figsize=(12, 4))
plt.plot(scores, marker="o")
plt.axhline(0, color="red", linestyle="--", alpha=0.5)
plt.title(f"Evolución del sentimiento — {SHOW_TITLE[:80]}")
plt.xlabel(f"Segmento (de {SEGMENTS})"); plt.ylabel("Compound sentiment")
plt.show()


## 9. TF-IDF: palabras distintivas por comediante

In [ ]:
# Agrupa por comediante (concatenando sus shows)
by_com = df.groupby("comedian")["transcript"].apply(lambda x: " ".join(x.fillna("")))
if len(by_com) >= 2:
    tfidf = TfidfVectorizer(stop_words=list(STOPWORDS), min_df=1, max_df=0.8, ngram_range=(1, 2))
    M = tfidf.fit_transform(by_com)
    vocab = tfidf.get_feature_names_out()

    rows = []
    for i, com in enumerate(by_com.index):
        top = pd.Series(M[i].toarray().ravel(), index=vocab).nlargest(8)
        rows.append({"comedian": com, "distinctive": ", ".join(top.index)})
    pd.DataFrame(rows).head(20)
else:
    print("Necesitas al menos 2 comediantes en el subset para TF-IDF.")
    pd.DataFrame()


## 10. Similaridad estilística entre comediantes

In [ ]:
if len(by_com) >= 2:
    sim = cosine_similarity(M)
    sim_df = pd.DataFrame(sim, index=by_com.index, columns=by_com.index)
    # Mostrar máximo 25 para que el heatmap sea legible
    top_sample = df["comedian"].value_counts().head(25).index
    show = sim_df.loc[sim_df.index.isin(top_sample), sim_df.columns.isin(top_sample)]
    plt.figure(figsize=(12, 10))
    sns.heatmap(show, cmap="viridis", square=True, cbar_kws={"label": "Coseno TF-IDF"})
    plt.title("Similaridad estilística entre comediantes")
    plt.tight_layout(); plt.show()


## 11. Generador de frases (cadena de Markov)

Modelo de bigramas: dada una palabra, elige la siguiente con
probabilidad proporcional a las co-ocurrencias en los transcripts del
subset. No es un LLM — es un juguete que captura giros típicos.


In [ ]:
from collections import defaultdict
import random

def build_markov(tokens_lists, order=2):
    model = defaultdict(Counter)
    for tokens in tokens_lists:
        for i in range(len(tokens) - order):
            key = tuple(tokens[i:i + order])
            model[key][tokens[i + order]] += 1
    return model

def generate(model, n_words=30, seed=None):
    if seed is None:
        seed = random.choice(list(model.keys()))
    out = list(seed)
    for _ in range(n_words):
        key = tuple(out[-len(seed):])
        nexts = model.get(key)
        if not nexts:
            break
        word = random.choices(list(nexts.keys()), weights=list(nexts.values()))[0]
        out.append(word)
    return " ".join(out)

# Usar tokens SIN filtrar stopwords para que las frases sean legibles
raw_tokens = df["transcript"].fillna("").map(
    lambda t: [w.lower() for w in TOKEN_RE.findall(t)]
).tolist()

model = build_markov(raw_tokens, order=2)
for _ in range(5):
    print("—", generate(model, n_words=25))


## 12. Cruce con IMDb — ¿qué predice el rating?

In [ ]:
sub = df.dropna(subset=["rating", "sentiment", "ttr", "word_count"])
print(f"Shows con rating: {len(sub)}")

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, col, label in zip(
    axes,
    ["sentiment", "ttr", "word_count"],
    ["Sentimiento", "Diversidad léxica (TTR)", "Longitud (# palabras)"],
):
    sns.regplot(data=sub, x=col, y="rating", ax=ax, scatter_kws={"alpha": 0.5})
    corr = sub[[col, "rating"]].corr().iloc[0, 1]
    ax.set_title(f"{label} vs rating (r={corr:.2f})")
plt.tight_layout(); plt.show()

print("\nCorrelaciones de Pearson con rating:")
sub[["sentiment", "ttr", "word_count", "word_count", "runtime_min", "votes"]].corrwith(sub["rating"]).sort_values()


## Siguientes pasos sugeridos

- **Topic modeling** (LDA/BERTopic) → temas latentes y cómo cambian con el tiempo.
- **Emociones discretas** (lexicon NRC) → 8 emociones por comediante, no solo polaridad.
- **Detección de catchphrases** → n-gramas con alta freq intra-comediante y baja inter-comediante.
- **Clustering de comediantes** (k-means sobre TF-IDF) + visualización UMAP/t-SNE.
- **Modelo predictivo del rating** (Ridge sobre TF-IDF + features) → qué palabras mueven la aguja.
- **Comparar productoras** (`distributors`): ¿Netflix vs HBO vs Comedy Central tienen perfiles distintos?
